# TP06 -  Intermediate Representations and Analysis with `Lark`

## Grammar Engineering

## MEI/2025-26

#### Nuno Macedo
Universidade do Minho

The following exercises are intended to be solved with pen and paper and implemented with a grammar processor for Python, [Lark](https://lark-parser.readthedocs.io).

# A simple imperative language

- Consider a simple language with
  - integer variable assignments
  - `if` branches (with optional `else`)
  - `while` loops
  - `print` statements

- Code blocks have mandatory brackets `{` `}`

- This language can be defined by the `Lark` grammar below

In [6]:
from lark import Lark, Transformer
grammar = """
?start: block

?block: stmt+

?stmt: assign_stmt ";"
     | if_stmt
     | while_stmt
     | for_stmt
     | print_stmt ";"
     | continue_stmt ";"
     | break_stmt ";"
     | return_stmt ";"

assign_stmt: NAME "=" expr

if_stmt: "if" "(" expr ")" "{" block "}" ("else" "{" block "}")?

while_stmt: "while" "(" expr ")" "{" block "}"

for_stmt: "for" "(" assign_stmt ";" expr ";" assign_stmt ")" "{" block "}"

print_stmt: "print" expr

return_stmt: "return" expr?

continue_stmt: "continue"
break_stmt: "break"

?expr: NAME -> var
     | NUMBER -> number
     | expr "+" expr -> add
     | expr "-" expr -> sub
     | expr "*" expr -> mul
     | expr "<" expr -> lt
     | expr ">" expr -> gt
     | "(" expr ")"

%import common.CNAME -> NAME
%import common.NUMBER
%import common.WS
%ignore WS
"""

parser = Lark(grammar)


# Exercise 1

Consider the simple imperative language defined above, and draw the CFG for the following code snippets.

## Exercise 1.1

In [ ]:
ex_1_1 = """
x = 1;
if (x < 10) {
    y = x + 1;
} else {
    y = 0;
}
print y;
"""

## Exercise 1.2



In [ ]:
ex_1_2 = """
x = 10;
y = x + 5;
if (y > 12) {
    z = x + y;
    z = 2 * z;
}
else {
    z = y - 2;
    print z;
}
while (z > 10) {
    z = z - 1;
    x = x + 2;
}
print z;
"""

"""
     (x = 10;)
        |
    (y = x + 5;)
        |
       (if)
        /\
(z = x+y; z=2*z)  (z=y-2; print z;)
    \   /
    (merge)
        |
    (z > 10)-----------------
   (     |                  |
    (z = z - 1;             |
    x = x + 2;)             |
        |                   |
    (merge)------------------
        |
    (print z;)
"""

## Exercise 1.3


In [ ]:
ex_1_3 = """
x = 10;
y = x + 5;
if (y > 12) {
    if (x < 20) {
        z = 2 * z;
    }
}
else {
    if (x < 20) {
        print x;
    } else {
      print x * 2;
    }
}
print 0;

"""

## Exercise 1.4


In [ ]:
ex_1_4 = """
x = 10;
while (x > 0) {
  if (x > 5) {
    print 5;
  } else {
    print 10;
  }
}
print 0;
"""

# `Lark` and IRs

- `Lark` already gives us fo free an intermediate representation (IR), the *abstract syntax tree* of the input text

- Using `Lark` tree traversals we can transform an AST into another IR, such as the *control-flow graph* (CFG)




## Extracting CFGs with `Lark` `Transformer`

- We can use a `Transformer` to create the CFG by composing it at each node of the AST:
  - get the CFGs corresponding to the children trees
  - connected their exit/entry into a new CFG

- We can create our own CFG class in Python

- Or we can use directly a standard library for graphs, such as [`NetworkX`](https://networkx.org/)


In [3]:
from lark import Lark, Transformer
import networkx as nx
from networkx.drawing.nx_pydot import write_dot

class CFGFragment:
    """A CFG fragment, a graph with an entry and an exit node."""
    def __init__(self, entry, exit, graph=None):
        self.entry = entry
        self.exit = exit
        self.graph = graph or nx.DiGraph()

    def merge(self, fragment):
        e, f = self.exit, fragment.entry
        self.graph.add_nodes_from(fragment.graph.nodes(data=True))
        self.graph.add_edges_from(fragment.graph.edges())
        self.graph.add_edge(e,f)
        self.exit = fragment.exit
        # merge previous exit with fragment entry since they'll be connected by a single line
        nx.contracted_nodes(self.graph,e,f,copy=False, self_loops=False)
        self.graph.nodes[e]["label"] = (self.graph.nodes[e]["label"] + "\n" + fragment.graph.nodes[f]["label"]).strip()
        # if the fragment had a single node, it was contracted into the previous exit
        if f == fragment.exit:
            self.exit = e

class CFGTransformer(Transformer):
    def __init__(self):
        self.counter = 0

    def new_block(self, stmt=""):
        name = f"B{self.counter}"
        self.counter += 1
        g = nx.DiGraph()
        g.add_node(name, label=stmt)
        return CFGFragment(name, name, g)

    def block(self, items):
        g = items[0]
        for i in items[1:]:
            g.merge(i)
        return g

    def assign_stmt(self, items):
        stmt_str = f"{items[0].value} = ..."
        return self.new_block(stmt_str)

    def print_stmt(self, items):
        stmt_str = "print ..."
        return self.new_block(stmt_str)

    def if_stmt(self, items):
        cond_frag = self.new_block("if ...")
        merge_frag = self.new_block("merge if")
        then_frag = items[1]

        cond_frag.graph.add_nodes_from(then_frag.graph.nodes(data=True))
        cond_frag.graph.add_edges_from(then_frag.graph.edges())
        cond_frag.graph.add_edge(cond_frag.exit, then_frag.entry)
        cond_frag.graph.add_edge(then_frag.exit, merge_frag.entry)

        if len(items) > 2:
            else_frag = items[2]
            cond_frag.graph.add_nodes_from(else_frag.graph.nodes(data=True))
            cond_frag.graph.add_edges_from(else_frag.graph.edges())
            cond_frag.graph.add_edge(cond_frag.exit, else_frag.entry)
            cond_frag.graph.add_edge(else_frag.exit, merge_frag.entry)
        else:
            cond_frag.graph.add_edge(cond_frag.exit, merge_frag.entry)


        cond_frag.graph.add_nodes_from(merge_frag.graph.nodes(data=True))

        return CFGFragment(cond_frag.entry, merge_frag.exit, cond_frag.graph)

    def while_stmt(self, items):
        cond_frag = self.new_block("while ...")
        body_frag = items[1]
        cond_frag.graph.add_nodes_from(body_frag.graph.nodes(data=True))
        cond_frag.graph.add_edges_from(body_frag.graph.edges())
        cond_frag.graph.add_edge(cond_frag.exit, body_frag.entry)

        merge_frag = self.new_block("merge while")
        cond_frag.graph.add_nodes_from(merge_frag.graph.nodes(data=True))
        cond_frag.graph.add_edge(cond_frag.exit, merge_frag.entry)
        cond_frag.graph.add_edge(body_frag.exit, cond_frag.entry)
        return CFGFragment(cond_frag.entry, merge_frag.exit, cond_frag.graph)


    def for_stmt(self, items):
        for_start = self.new_block("for ...")
        body_frag = items[3]
        for_start.graph.add_nodes_from(body_frag.graph.nodes(data=True))
        for_start.graph.add_edges_from(body_frag.graph.edges())
        for_start.graph.add_edge(for_start.exit, body_frag.entry)

        for_merge = self.new_block("merge for")
        for_start.graph.add_nodes_from(for_merge.graph.nodes(data=True))
        for_start.graph.add_edge(for_start.exit, for_merge.entry)
        for_start.graph.add_edge(body_frag.exit, for_start.entry)
        return CFGFragment(for_start.entry, for_merge.exit, for_start.graph)

In [4]:
!apt-get -qq install graphviz > /dev/null
!pip install graphviz pydot

E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?


In [4]:
from networkx.drawing.nx_pydot import to_pydot
from graphviz import Source

# parse
tree = parser.parse(ex_1_4)

# criar CFG
cfg_transformer = CFGTransformer()
cfg = cfg_transformer.transform(tree).graph

# converter para pydot / graphviz
dot_str = to_pydot(cfg).to_string()
src = Source(dot_str)

# a última linha deve ser apenas o objeto para renderizar
src


NameError: name 'parser' is not defined

# Exercise 2

Extend the language above, the corresponding grammar, and the CFG extraction `Transformer`, to support additional control flow constructs.

## Exercise 2.1

Introduce in the language `for` loops of the shape:

```
for (i = 10; i < 10; i = i + 1) { ... }
```


## Exercise 2.2


Introduce in the language additional control flow constructs statements, namely `break` and `continue` statements for loops:

```
while (z < 10) {
  if (z == 0) {
    break;
  } else {
    if (z > 10) {
      continue;
    }
  }
  z = z - 1;
  print z;
}
```

## Exercise 2.3

Introduce in the language `return` statements that immediately terminate a procedure:

```
x = 10;
if (x > 20) {
  return x;
}
while (x > 10) {
  x = x / 2;
  if (x == 0) {
    return 0;
  }
}
```


# Exercise 3

Use the CFG extracted from source code to perform basic static analyses.

*Hint*: explore the graph traversals provided by `NetworkX`.



## Exercise 3.1

Use the CFG to detect *dead code*, i.e., blocks that are not *reachable*. Assume the language extension that contains at least `return` statements.


In [ ]:
dead_code = """
x = 10;
y = x + 5;
if (y > 12) {
    if (x < 20) {
        return z;
        if (x < 10) {
            z = 2 * z;
        }
    }
}
else {
    if (x < 20) {
        print x;
    } else {
      print x * 2;
    }
}
return x;
print 0;
"""

tree = parser.parse(code)
cfg_transformer = CFGTransformer()
entry = cfg_transformer.transform(tree)


def find_dead_blocks(graph, entry):
    """
    >>> find_dead_blocks(entry.graph, entry.entry)
    ['print ...', 'if ...', 'merge', 'z = ...']
    """

    return []

import doctest
doctest.testmod()

## Exercise 3.2

Use the CFG to detect *back loops*, i.e., transitions that loop back into a previously visited node.


In [ ]:
code = """
x = 10;
y = x + 5;
while (y > 12) {
    if (x < 20) {
        if (x < 10) {
            z = 2 * z;
        }
    }
}
return x;
print 0;
"""

tree = parser.parse(code)
cfg_transformer = CFGTransformer()
entry = cfg_transformer.transform(tree)


def find_back_edges(G, entry):
    """
    >>> find_back_edges(entry.graph, entry.entry)
    [('merge', 'x = ...\ny = ...\nwhile ...')]
    """

    return []

import doctest
doctest.testmod()

-- Nuno Macedo